# 01 — Data Cleaning

Tujuan notebook ini:
1. Generate dataset sintetis pelanggan telco (atau load jika sudah ada).
2. Eksplorasi awal: struktur, tipe data, missing values, duplikat.
3. Menjalankan pipeline cleaning dari `src.data_processing`.
4. Validasi hasil cleaning sebelum lanjut ke EDA.

Semua logika cleaning di-encapsulate dalam `src/data_processing.py` agar
dapat di-reuse oleh notebook lain dan oleh Streamlit app (single source of truth).


In [14]:
import sys
from pathlib import Path

# Tambahkan root project ke sys.path supaya `from src...` bisa di-import
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from src.data_processing import (
    generate_synthetic_data,
    save_raw_data,
    load_raw_data,
    clean_data,
    save_clean_data,
)
from src.utils import RAW_DATA_PATH, CLEAN_DATA_PATH

pd.set_option('display.max_columns', None)
print(f"Project root : {PROJECT_ROOT}")
print(f"Raw data path : {RAW_DATA_PATH}")
print(f"Clean data path: {CLEAN_DATA_PATH}")


Project root : c:\Users\ACER\Downloads\telco-churn-2026\telco-churn-2026
Raw data path : C:\Users\ACER\Downloads\telco-churn-2026\telco-churn-2026\data\raw\telco_customer_churn.csv
Clean data path: C:\Users\ACER\Downloads\telco-churn-2026\telco-churn-2026\data\processed\telco_clean.csv


## 1. Generate / Load Raw Data


In [15]:
# Generate dataset baru atau load jika file raw sudah ada
if RAW_DATA_PATH.exists():
    print("Raw data sudah ada, loading dari disk")
    df_raw = load_raw_data()
else:
    print("Generating dataset sintetis baru")
    df_raw = generate_synthetic_data(n=7000, random_state=42)
    save_raw_data(df_raw)

print(f"\nShape: {df_raw.shape}")
df_raw.head()


Raw data sudah ada, loading dari disk

Shape: (7010, 18)


,customer_id,gender,usia,kota,tenure_bulan,paket,jenis_jaringan,kuota_gb,biaya_bulanan,punya_streaming_bundle,punya_ewallet_linked,metode_bayar,jumlah_komplain_6bln,skor_kepuasan_csat,pernah_upgrade_paket,total_pemakaian_data_gb,frekuensi_login_app,churn
0,CUST-00000,Male,60,Yogyakarta,19,Prabayar,5G,25,146.01,Yes,Yes,Transfer Bank,2,5.0,No,14.74,11,0
1,CUST-00001,Female,37,Lainnya,41,Hybrid,5G,50,90.54,Yes,Yes,E-Wallet,2,3.0,Yes,14.25,6,0
2,CUST-00002,Female,63,Semarang,69,Prabayar,Fiber,5,133.45,No,No,E-Wallet,0,4.0,No,4.63,7,0
3,CUST-00003,Male,44,Medan,51,Pascabayar,Fiber,Unlimited,224.06,No,No,Auto-debit,1,3.0,No,31.71,10,0
4,CUST-00004,Male,36,Lainnya,22,Prabayar,4G,50,67.05,Yes,No,E-Wallet,1,4.0,Yes,11.88,9,0


## 2. Eksplorasi Awal: Struktur & Tipe Data


In [16]:
print("     INFO       \n")
df_raw.info()

     INFO       

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7010 entries, 0 to 7009
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customer_id              7010 non-null   object 
 1   gender                   7010 non-null   object 
 2   usia                     7010 non-null   int64  
 3   kota                     7010 non-null   object 
 4   tenure_bulan             7010 non-null   int64  
 5   paket                    7010 non-null   object 
 6   jenis_jaringan           7010 non-null   object 
 7   kuota_gb                 7010 non-null   object 
 8   biaya_bulanan            7010 non-null   float64
 9   punya_streaming_bundle   7010 non-null   object 
 10  punya_ewallet_linked     7010 non-null   object 
 11  metode_bayar             7010 non-null   object 
 12  jumlah_komplain_6bln     7010 non-null   int64  
 13  skor_kepuasan_csat       6870 non-null   float64
 14  pernah

In [17]:
print("        STATISTIK DESKRIPTIF (numerik)       \n")
df_raw.describe()


        STATISTIK DESKRIPTIF (numerik)       



,usia,tenure_bulan,biaya_bulanan,jumlah_komplain_6bln,skor_kepuasan_csat,total_pemakaian_data_gb,frekuensi_login_app,churn
count,7010.000000,7010.000000,7010.000000,7010.000000,6870.000000,6905.000000,7010.000000,7010.000000
mean,40.881740,36.044650,149.789194,0.805136,3.015429,15.133206,8.033238,0.152782
std,13.448172,20.929776,66.921408,0.895751,1.401929,15.257108,2.805210,0.359803
min,18.000000,0.000000,12.540000,0.000000,1.000000,0.000000,0.000000,0.000000
25%,29.000000,18.000000,100.727500,0.000000,2.000000,4.440000,6.000000,0.000000
50%,41.000000,36.000000,139.730000,1.000000,3.000000,10.360000,8.000000,0.000000
75%,52.000000,54.000000,188.545000,1.000000,4.000000,21.000000,10.000000,0.000000
max,64.000000,71.000000,569.990000,6.000000,5.000000,155.020000,21.000000,1.000000


In [18]:
print("        STATISTIK DESKRIPTIF (kategorikal)       \n")
df_raw.describe(include='object')


        STATISTIK DESKRIPTIF (kategorikal)       



,customer_id,gender,kota,paket,jenis_jaringan,kuota_gb,punya_streaming_bundle,punya_ewallet_linked,metode_bayar,pernah_upgrade_paket
count,7010,7010,7010,7010,7010,7010,7010,7010,7010,7010
unique,7000,2,8,3,3,6,2,2,4,2
top,CUST-04102,Male,Jakarta,Prabayar,4G,5,No,Yes,Transfer Bank,No
freq,2,3557,1772,3474,3087,1233,4212,4164,1796,4849


## 3. Deteksi Missing Values


In [19]:
missing = df_raw.isna().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

missing_df = pd.DataFrame({
    'jumlah_missing': missing,
    'persentase (%)': missing_pct
})
missing_df = missing_df[missing_df['jumlah_missing'] > 0]
print(missing_df)


                         jumlah_missing  persentase (%)
skor_kepuasan_csat                  140             2.0
total_pemakaian_data_gb             105             1.5


## 4. Cek Duplikat & Kolom Bermasalah (`kuota_gb`)

- `customer_id` unik, cek apakah ada duplikat.
- Kolom `kuota_gb` adalah **mixed-type**: berisi angka (5, 10, 25, ...) DAN
  string `'Unlimited'`.  perlu ditangani sebelum modeling.


In [20]:
n_dup = df_raw['customer_id'].duplicated().sum()
print(f"Jumlah customer_id duplikat: {n_dup}")

if n_dup > 0:
    display_cols = ['customer_id', 'gender', 'usia', 'kota', 'tenure_bulan']
    print("\nContoh baris duplikat:")
    dup_ids = df_raw[df_raw['customer_id'].duplicated(keep=False)]['customer_id'].unique()[:3]
    print(df_raw[df_raw['customer_id'].isin(dup_ids)][display_cols].sort_values('customer_id'))


Jumlah customer_id duplikat: 10

Contoh baris duplikat:
     customer_id  gender  usia        kota  tenure_bulan
647   CUST-00647  Female    19     Bandung            24
7005  CUST-00647  Female    19     Bandung            24
1397  CUST-01397  Female    26  Yogyakarta            33
7000  CUST-01397  Female    26  Yogyakarta            33
3075  CUST-03075  Female    26  Yogyakarta             2
7004  CUST-03075  Female    26  Yogyakarta             2


In [21]:
print("Unique values kolom 'kuota_gb':")
print(df_raw['kuota_gb'].unique())
print(f"\nTipe data kolom 'kuota_gb': {df_raw['kuota_gb'].dtype}")
print("\n > Kolom ini mixed-type (angka + 'Unlimited'), perlu dinormalisasi")
print("   sebelum bisa dipakai sebagai fitur numerik di model.")


Unique values kolom 'kuota_gb':
['25' '50' '5' 'Unlimited' '10' '100']

Tipe data kolom 'kuota_gb': object

 > Kolom ini mixed-type (angka + 'Unlimited'), perlu dinormalisasi
   sebelum bisa dipakai sebagai fitur numerik di model.


## 5. Preview Outlier pada `biaya_bulanan`

cek distribusi `biaya_bulanan` menggunakan metode IQR untuk melihat
apakah ada nilai ekstrem yang perlu di-cap.


In [22]:
q1, q3 = df_raw['biaya_bulanan'].quantile([0.25, 0.75])
iqr = q3 - q1
lower = max(0, q1 - 1.5 * iqr)
upper = q3 + 1.5 * iqr

n_outliers = ((df_raw['biaya_bulanan'] < lower) | (df_raw['biaya_bulanan'] > upper)).sum()

print(f"Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}")
print(f"Lower bound={lower:.2f}, Upper bound={upper:.2f}")
print(f"Jumlah outlier: {n_outliers} ({n_outliers/len(df_raw)*100:.2f}%)")
print(f"\nMax biaya_bulanan saat ini: {df_raw['biaya_bulanan'].max():.2f}")


Q1=100.73, Q3=188.55, IQR=87.82
Lower bound=0.00, Upper bound=320.27
Jumlah outlier: 133 (1.90%)

Max biaya_bulanan saat ini: 569.99


## 6. Menjalankan Pipeline Cleaning

Fungsi `clean_data()` dari `src.data_processing` menangani isu:

1. Drop duplikat berdasarkan `customer_id` (keep first).
2. Normalisasi `kuota_gb` : konversi `'Unlimited'` > 999, buat flag
   `is_unlimited`, cast ke numeric.
3. Imputasi missing values:
   - `skor_kepuasan_csat` > median per grup `paket`.
   - `total_pemakaian_data_gb` > median global.
4. Outlier capping pada `biaya_bulanan` menggunakan IQR.
5. Cast tipe data: kolom kategorikal > `category`, target > `int`.

Log proses akan tercetak otomatis (level INFO) untuk traceability.


In [23]:
df_clean = clean_data(df_raw)
print(f"\nShape sebelum cleaning: {df_raw.shape}")
print(f"Shape setelah cleaning : {df_clean.shape}")
df_clean.head()


[2026-06-28 21:05:07] INFO | src.data_processing | Duplikat dihapus: 10 baris (7010 -> 7000)
[2026-06-28 21:05:07] INFO | src.data_processing | Kolom kuota_gb dinormalisasi. is_unlimited rate=15.77%
[2026-06-28 21:05:07] INFO | src.data_processing | Missing skor_kepuasan_csat diisi: 140 nilai (median per grup paket)
[2026-06-28 21:05:07] INFO | src.data_processing | Missing total_pemakaian_data_gb diisi: 105 nilai (median global)
[2026-06-28 21:05:07] INFO | src.data_processing | Tidak ada missing values setelah imputasi.
[2026-06-28 21:05:07] INFO | src.data_processing | Outlier biaya_bulanan di-cap: 132 baris (bounds=[0.00, 320.23])
[2026-06-28 21:05:07] INFO | src.data_processing | Cleaning selesai. Final shape=(7000, 19)



Shape sebelum cleaning: (7010, 18)
Shape setelah cleaning : (7000, 19)


,customer_id,gender,usia,kota,tenure_bulan,paket,jenis_jaringan,kuota_gb,biaya_bulanan,punya_streaming_bundle,punya_ewallet_linked,metode_bayar,jumlah_komplain_6bln,skor_kepuasan_csat,pernah_upgrade_paket,total_pemakaian_data_gb,frekuensi_login_app,churn,is_unlimited
0,CUST-00000,Male,60,Yogyakarta,19,Prabayar,5G,25,146.01,Yes,Yes,Transfer Bank,2,5,No,14.74,11,0,0
1,CUST-00001,Female,37,Lainnya,41,Hybrid,5G,50,90.54,Yes,Yes,E-Wallet,2,3,Yes,14.25,6,0,0
2,CUST-00002,Female,63,Semarang,69,Prabayar,Fiber,5,133.45,No,No,E-Wallet,0,4,No,4.63,7,0,0
3,CUST-00003,Male,44,Medan,51,Pascabayar,Fiber,999,224.06,No,No,Auto-debit,1,3,No,31.71,10,0,1
4,CUST-00004,Male,36,Lainnya,22,Prabayar,4G,50,67.05,Yes,No,E-Wallet,1,4,Yes,11.88,9,0,0


## 7. Validasi Hasil Cleaning

Pastikan:
- Tidak ada missing values.
- `customer_id` sudah unik.
- `kuota_gb` sudah numeric & ada kolom `is_unlimited`.
- `biaya_bulanan` sudah dicap (max <= upper bound).
- Tipe data kategorikal.


In [24]:
print("=== VALIDASI ===")
print(f"1. Total missing values     : {df_clean.isna().sum().sum()}")
print(f"2. customer_id unik?        : {df_clean['customer_id'].is_unique}")
print(f"3. kuota_gb dtype            : {df_clean['kuota_gb'].dtype}")
print(f"4. is_unlimited ada?         : {'is_unlimited' in df_clean.columns}")
print(f"5. Max biaya_bulanan         : {df_clean['biaya_bulanan'].max():.2f} (upper bound={upper:.2f})")
print(f"6. churn dtype               : {df_clean['churn'].dtype}")
print(f"\n      DTYPES SETELAH CLEANING     ")
print(df_clean.dtypes)


=== VALIDASI ===
1. Total missing values     : 0
2. customer_id unik?        : True
3. kuota_gb dtype            : int64
4. is_unlimited ada?         : True
5. Max biaya_bulanan         : 320.23 (upper bound=320.27)
6. churn dtype               : int64

      DTYPES SETELAH CLEANING     
customer_id                  object
gender                     category
usia                          int64
kota                       category
tenure_bulan                  int64
paket                      category
jenis_jaringan             category
kuota_gb                      int64
biaya_bulanan               float64
punya_streaming_bundle     category
punya_ewallet_linked       category
metode_bayar               category
jumlah_komplain_6bln          int64
skor_kepuasan_csat            int64
pernah_upgrade_paket       category
total_pemakaian_data_gb     float64
frekuensi_login_app           int64
churn                         int64
is_unlimited                  int64
dtype: object


In [25]:
print("     DISTRIBUSI TARGET (churn)    \n")
churn_counts = df_clean['churn'].value_counts()
churn_pct = df_clean['churn'].value_counts(normalize=True) * 100

summary = pd.DataFrame({'jumlah': churn_counts, 'persentase (%)': churn_pct.round(2)})
print(summary)
print(f"\n-> Dataset IMBALANCED: churn rate {churn_pct[1]:.2f}%.")
print("   Perlu ditangani saat modeling (lihat notebook 04_modeling, scale_pos_weight).")


     DISTRIBUSI TARGET (churn)    

       jumlah  persentase (%)
churn                        
0        5930           84.71
1        1070           15.29

-> Dataset IMBALANCED: churn rate 15.29%.
   Perlu ditangani saat modeling (lihat notebook 04_modeling, scale_pos_weight).


## 8. Simpan Hasil Cleaning

* Data bersih disimpan `data/processed/telco_clean.csv`
* menjadi input untuk notebook `02_eda.ipynb` dan `03_feature_engineering.ipynb`.


In [26]:
save_clean_data(df_clean)
print(f"Data bersih disimpan ke: {CLEAN_DATA_PATH}")
print(f"Shape final: {df_clean.shape}")


[2026-06-28 21:05:08] INFO | src.data_processing | Clean data saved to C:\Users\ACER\Downloads\telco-churn-2026\telco-churn-2026\data\processed\telco_clean.csv (shape=(7000, 19))


Data bersih disimpan ke: C:\Users\ACER\Downloads\telco-churn-2026\telco-churn-2026\data\processed\telco_clean.csv
Shape final: (7000, 19)


## Ringkasan & Next Steps

| Isu | Status |
|---|---|
| Duplikat `customer_id` | ✅ Dihapus |
| Mixed-type `kuota_gb` | ✅ Dinormalisasi + flag `is_unlimited` |
| Missing values | ✅ Diimputasi (median per grup / global) |
| Outlier `biaya_bulanan` | ✅ Di-cap dengan IQR |
| Tipe data | ✅ Sesuai (category, int, float) |

**Insight awal**: Dataset imbalanced dengan churn rate ~15%. Hal ini akan
menjadi pertimbangan utama saat memilih metrik evaluasi (Recall & ROC-AUC
lebih relevan daripada Accuracy) dan teknik penanganan imbalance
(`scale_pos_weight` pada XGBoost).

**Lanjut ke**: `02_eda.ipynb` untuk eksplorasi pola & hubungan antar variabel.
